# STAIR-Enhanced v2a: Residual-Whitening Projector (ResOnly)

**Mục tiêu:** Thực nghiệm Micro-Ablation Study v2a — Giữ nguyên SVD Whitening tĩnh (Structural Prior đóng băng) và kết hợp thêm nhánh thặng dư phi tuyến `ResidualWhiteningProjector` với cơ chế Warm-start SVD và Dual-Optimizer.

$$\mathbf{e}_i^{final} = \mathbf{E}_{svd, i} + \lambda_{res} \cdot \Delta_i$$

**Cấu hình Pipeline:**
- **Mô hình:** `EnhancedSTAIR_v2a` (`main_enhanced_v2.py`)
- **Projector:** `ResidualWhiteningProjector` (`models/residual_projector_v2.py`)
- **Dual Optimizer:** `AdamWSEvo` (User/Item embeddings, lr=1e-3) + `torch.optim.Adam` (res_projector, lr=5e-3)
- **AMP:** Tự động bật `torch.cuda.amp.autocast()` và `GradScaler()`
- **Datasets:** Amazon 2014 MMRec (**Baby**, **Sports**, **Electronics**)

| Tập dữ liệu | Baseline R@10 | Baseline R@20 | Baseline N@10 | Baseline N@20 | v1 R@20 | v1 N@20 |
|---|:---:|:---:|:---:|:---:|:---:|:---:|
| **Baby** | 0.0674 | 0.1042 | 0.0359 | 0.0454 | 0.0948 (−9.0%) | 0.0412 (−9.3%) |
| **Sports** | 0.0743 | 0.1111 | 0.0405 | 0.0500 | 0.1040 (−6.4%) | 0.0466 (−6.8%) |
| **Electronics** | 0.0442 | 0.0665 | 0.0246 | 0.0303 | 0.0601 (−9.6%) | 0.0274 (−9.6%) |


## Cell 1 — Thiết lập Môi trường & Cài đặt STAIR-Enhanced


In [ ]:
# Cell 1: Môi trường & Cài đặt Dependencies
import os, shutil, subprocess, sys

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
os.chdir('/kaggle/working')

# 1. Luôn clone mới nhất từ repository để đảm bảo có đầy đủ file mã nguồn mới nhất
if os.path.exists(STAIR_DIR):
    print('Làm sạch thư mục cũ để clone mới nhất...')
    shutil.rmtree(STAIR_DIR)

print('Cloning STAIR-Enhanced repository (branch main)...')
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ThanhChuong12/STAIR-Enhanced.git', STAIR_DIR], check=True)

# 2. Thêm STAIR_DIR vào sys.path để python luôn tìm thấy package `models`, `optimizers`
for p in [STAIR_DIR, '/kaggle/working']:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(STAIR_DIR)

# 3. Cài đặt các thư viện cần thiết
print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torchdata==0.7.1'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'freerec==0.8.5', 'nvidia-ml-py', 'prettytable', 'matplotlib'], check=True)

import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_TAG  = 'cu' + torch.version.cuda.replace('.','') if torch.cuda.is_available() else 'cpu'
print(f'Installing torch-geometric for torch={TORCH_VER}+{CUDA_TAG}...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric', '-f', f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html'], check=False)

import freerec
print('=' * 60)
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB')
print(f'FreeRec : {freerec.__version__}')
print(f'sys.path: {sys.path[:2]}')
print('=' * 60)
print('[OK] Environment setup complete!')


## Cell 2 — Chuẩn bị Dữ liệu từ Kaggle Input (Tự động quét)


In [ ]:
# Cell 2: Chuẩn bị dữ liệu từ Kaggle Input sang /kaggle/data & STAIR-Enhanced/data
import os, shutil

DATA_ROOTS = ['/kaggle/data', '/kaggle/working/STAIR-Enhanced/data']
for root in DATA_ROOTS:
    os.makedirs(root, exist_ok=True)

REQUIRED_EXTENSIONS = ('.npy', '.pkl', '.txt', '.inter', '.item')

def copy_dataset(keywords, full_name):
    copied_files = 0
    for root_dir, _, files in os.walk('/kaggle/input'):
        if any(kw.lower() in root_dir.lower() for kw in keywords):
            for f in files:
                if f.endswith(REQUIRED_EXTENSIONS):
                    src_path = os.path.join(root_dir, f)
                    for target_root in DATA_ROOTS:
                        dest_dir = os.path.join(target_root, full_name)
                        os.makedirs(dest_dir, exist_ok=True)
                        shutil.copy(src_path, os.path.join(dest_dir, f))
                    copied_files += 1
    print(f'[OK] {full_name}: {copied_files} files copied.')

copy_dataset(['baby', 'amazon2014baby'], 'Amazon2014Baby_550_MMRec')
copy_dataset(['sports', 'amazon2014sports'], 'Amazon2014Sports_550_MMRec')
copy_dataset(['electronics', 'amazon2014electronics'], 'Amazon2014Electronics_550_MMRec')

print(f'\nDữ liệu sẵn sàng tại: {DATA_ROOTS[0]}')


## Cell 3 — Kiểm tra Module v2a & Warm-start


In [ ]:
# Cell 3: Kiểm tra cấu hình và Modules v2a
import sys, os, math, torch, yaml

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
for p in [STAIR_DIR, '/kaggle/working']:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(STAIR_DIR)

# 1. Test Import Module v2
try:
    from models.residual_projector_v2 import ResidualWhiteningProjector, composite_embeddings
except ModuleNotFoundError:
    from models import ResidualWhiteningProjector, composite_embeddings

print('[1/3] Module ResidualWhiteningProjector import thành công!')

# 2. Sanity check Warm-start & Residual logic
N_TEST = 100
proj = ResidualWhiteningProjector(d_text=384, d_visual=4096, d_hidden=64, lambda_init=0.1)
t_feat = torch.randn(N_TEST, 384)
v_feat = torch.randn(N_TEST, 4096)
e_svd  = torch.randn(N_TEST, 64).mul(math.sqrt(N_TEST / 64))

proj.init_warm_start_weights(t_feat, v_feat)
with torch.no_grad():
    delta = proj(t_feat, v_feat)
    e_final = composite_embeddings(proj, t_feat, v_feat, e_svd, N_TEST, 64)

print(f'[2/3] Sanity Check:')
print(f'      Delta shape   : {tuple(delta.shape)} (L2 norm: {delta.norm(dim=-1).mean():.4f})')
print(f'      e_final shape : {tuple(e_final.shape)} (Scale: {e_final.norm(dim=-1).mean():.4f})')
print(f'      lambda_res    : {proj.lambda_res.item():.4f} (requires_grad={proj.lambda_res.requires_grad})')

# 3. Kiểm tra YAML configs
configs = [
    f'{STAIR_DIR}/configs/Amazon2014Baby_550_MMRec.yaml',
    f'{STAIR_DIR}/configs/Amazon2014Sports_550_MMRec.yaml',
    f'{STAIR_DIR}/configs/Amazon2014Electronics_550_MMRec.yaml',
]
print('[3/3] Kiểm tra file cấu hình YAML:')
for cfg_path in configs:
    if os.path.exists(cfg_path):
        with open(cfg_path) as f:
            data = yaml.safe_load(f)
        print(f'  - {os.path.basename(cfg_path)}: epochs={data.get("epochs")}, lr={data.get("lr")}, monitors={data.get("monitors")}')
    else:
        print(f'  [ERR] Không tìm thấy {cfg_path}')


## Cell 4 — Helper Functions & Training Runner


In [ ]:
# Cell 4: Hàm hỗ trợ chạy Training & Giám sát Phần cứng
import re, os, time, threading, subprocess, sys
import pynvml

all_logs = {'baby': {'vram': [], 'time': []}, 'sports': {'vram': [], 'time': []}, 'electronics': {'vram': [], 'time': []}}
profiling_active = False
current_ds_key   = None

def hardware_profiler(interval=2.0):
    try:
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        t0 = time.time()
        while profiling_active:
            mem = pynvml.nvmlDeviceGetMemoryInfo(handle)
            if current_ds_key in all_logs:
                all_logs[current_ds_key]['vram'].append(mem.used / 1024**2)
                all_logs[current_ds_key]['time'].append(time.time() - t0)
            time.sleep(interval)
        pynvml.nvmlShutdown()
    except Exception:
        pass

def extract_best_test(log_path):
    if not os.path.exists(log_path):
        return None, None
    with open(log_path, encoding='utf-8', errors='ignore') as f:
        flat = f.read().replace('\n', ' ')
    ep_m = re.search(r'Load best model @Epoch:\s*(\d+)', flat)
    best_ep = int(ep_m.group(1)) if ep_m else None
    m = re.search(
        r'Load best model @Epoch.*?TEST.*?RECALL@10 Avg:\s*([0-9.]+).*?'
        r'RECALL@20 Avg:\s*([0-9.]+).*?NDCG@10 Avg:\s*([0-9.]+).*?NDCG@20 Avg:\s*([0-9.]+)',
        flat
    )
    if m:
        return best_ep, dict(zip(['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20'], [float(x) for x in m.groups()]))
    return best_ep, None

def parse_lambda_res_history(log_path):
    if not os.path.exists(log_path): return []
    with open(log_path, encoding='utf-8', errors='ignore') as f:
        content = f.read()
    matches = re.findall(r'\[lambda_res @epoch\s*(\d+)\]:\s*mean=([0-9.]+)', content)
    return [(int(ep), float(val)) for ep, val in matches]

def run_training(key, yaml_cfg, data_root, log_path, extra_args=None):
    global profiling_active, current_ds_key
    print(f'\n{"="*60}')
    print(f'BẮT ĐẦU HUẤN LUYỆN v2a (ResOnly): {key.upper()}')
    print(f'Config : {yaml_cfg}')
    print(f'Log    : {log_path}')
    print('='*60)

    current_ds_key = key
    profiling_active = True
    prof_thread = threading.Thread(target=hardware_profiler, args=(2.0,), daemon=True)
    prof_thread.start()

    cmd = [sys.executable, 'main_enhanced_v2.py', '--config', yaml_cfg, '--root', data_root]
    if extra_args:
        cmd.extend(extra_args)

    t0 = time.time()
    with open(log_path, 'w', encoding='utf-8') as logf:
        result = subprocess.run(cmd, stdout=logf, stderr=subprocess.STDOUT, cwd='/kaggle/working/STAIR-Enhanced')
    elapsed = time.time() - t0

    profiling_active = False
    prof_thread.join(timeout=5)

    if result.returncode != 0:
        print(f'[THẤT BẠI] Mã lỗi {result.returncode} (Thời gian: {elapsed/60:.1f} phút)')
        with open(log_path, errors='replace') as f:
            print('30 dòng log cuối cùng:')
            print(''.join(f.readlines()[-30:]))
    else:
        print(f'[HOÀN THÀNH] {key.upper()} trong {elapsed/60:.1f} phút')
        ep, metrics = extract_best_test(log_path)
        if metrics:
            print(f'  - Best checkpoint @Epoch: {ep}')
            for k, v in metrics.items():
                print(f'    * {k}: {v:.6f}')
    return result.returncode


## Cell 5 — Huấn luyện v2a trên Baby & Sports


In [ ]:
# Cell 5: Huấn luyện STAIR-Enhanced v2a trên Baby & Sports
import torch, os

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
DATA_ROOT = '/kaggle/data'
LOG_DIR   = '/kaggle/working/logs_enhanced_v2a'
os.makedirs(LOG_DIR, exist_ok=True)

# 1. Huấn luyện Baby
run_training(
    key       = 'baby',
    yaml_cfg  = f'{STAIR_DIR}/configs/Amazon2014Baby_550_MMRec.yaml',
    data_root = DATA_ROOT,
    log_path  = f'{LOG_DIR}/baby.log',
)
torch.cuda.empty_cache()

# 2. Huấn luyện Sports
run_training(
    key       = 'sports',
    yaml_cfg  = f'{STAIR_DIR}/configs/Amazon2014Sports_550_MMRec.yaml',
    data_root = DATA_ROOT,
    log_path  = f'{LOG_DIR}/sports.log',
)
torch.cuda.empty_cache()

print('=' * 60)
print('✅ HOÀN THÀNH HUẤN LUYỆN TẬP BABY & SPORTS (v2a)!')
print('=' * 60)


## Cell 6 — Huấn luyện v2a trên Electronics (Tập lớn nhất)


In [ ]:
# Cell 6: Huấn luyện STAIR-Enhanced v2a trên tập Electronics (~1.7M tương tác)
# Chạy riêng biệt ở cell này để quản lý thời gian trên Kaggle GPU (T4 ước tính 4-5 tiếng)
import torch, os

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
DATA_ROOT = '/kaggle/data'
LOG_DIR   = '/kaggle/working/logs_enhanced_v2a'
os.makedirs(LOG_DIR, exist_ok=True)

run_training(
    key       = 'electronics',
    yaml_cfg  = f'{STAIR_DIR}/configs/Amazon2014Electronics_550_MMRec.yaml',
    data_root = DATA_ROOT,
    log_path  = f'{LOG_DIR}/electronics.log',
)
torch.cuda.empty_cache()

print('=' * 60)
print('✅ HOÀN THÀNH HUẤN LUYỆN TẬP ELECTRONICS (v2a)!')
print('=' * 60)


## Cell 7 — Tổng hợp Kết quả & So sánh Ablation Study (Baseline vs v1 vs v2a)


In [ ]:
# Cell 7: Bảng so sánh Ablation Study: Baseline vs v1 vs v2a
from prettytable import PrettyTable
import os, math

LOG_DIR_V2 = '/kaggle/working/logs_enhanced_v2a'

BASELINE = {
    'baby':        {'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
    'sports':      {'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
    'electronics': {'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
}

V1_RESULTS = {
    'baby':        {'Recall@10': 0.0611, 'Recall@20': 0.0948, 'NDCG@10': 0.0325, 'NDCG@20': 0.0412},
    'sports':      {'Recall@10': 0.0695, 'Recall@20': 0.1040, 'NDCG@10': 0.0376, 'NDCG@20': 0.0466},
    'electronics': {'Recall@10': 0.0401, 'Recall@20': 0.0601, 'NDCG@10': 0.0223, 'NDCG@20': 0.0274},
}

v2_results = {}
for ds in ['baby', 'sports', 'electronics']:
    log = os.path.join(LOG_DIR_V2, f'{ds}.log')
    ep, metrics = extract_best_test(log)
    v2_results[ds] = {'epoch': ep, 'metrics': metrics}

METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

print('=' * 95)
print('BẢNG SO SÁNH ABLATION STUDY: STAIR Baseline vs Enhanced v1 vs Enhanced v2a (ResOnly)')
print('=' * 95)

for ds in ['baby', 'sports', 'electronics']:
    t = PrettyTable()
    t.field_names = ['Chỉ số', 'STAIR Baseline', 'Enhanced v1 (Replace)', 'Enhanced v2a (Residual)', 'Δ(v2a vs Baseline)']
    t.align = 'r'; t.align['Chỉ số'] = 'l'
    bl  = BASELINE[ds]
    v1  = V1_RESULTS[ds]
    v2  = v2_results[ds]['metrics'] or {}
    for m in METRICS:
        bl_val  = bl.get(m, float('nan'))
        v1_val  = v1.get(m, float('nan'))
        v2_val  = v2.get(m, float('nan'))
        delta   = f"{(v2_val - bl_val)/bl_val*100:+.2f}%" if (bl_val and v2_val and not math.isnan(v2_val)) else 'N/A'
        t.add_row([
            m,
            f'{bl_val:.4f}',
            f'{v1_val:.4f}' if v1_val else 'N/A',
            f'{v2_val:.4f}' if (v2_val and not math.isnan(v2_val)) else 'N/A',
            delta
        ])
    print(f'\nTập dữ liệu: {ds.upper()} (Best Epoch v2a: {v2_results[ds]["epoch"]})')
    print(t)
print('=' * 95)


## Cell 8 — Biểu đồ lambda_res & Đường cong Học tập (Learning Curves)


In [ ]:
# Cell 8: Vẽ biểu đồ lambda_res Evolution & VALID Curves
import matplotlib.pyplot as plt, re, os

LOG_DIR = '/kaggle/working/logs_enhanced_v2a'
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('STAIR-Enhanced v2a: lambda_res Evolution qua các Epochs', fontsize=14, fontweight='bold')

for i, ds in enumerate(['baby', 'sports', 'electronics']):
    log_path = os.path.join(LOG_DIR, f'{ds}.log')
    history = parse_lambda_res_history(log_path)
    ax = axes[i]
    if history:
        epochs = [h[0] for h in history]
        lambdas = [h[1] for h in history]
        ax.plot(epochs, lambdas, 'b-', linewidth=2, label='lambda_res')
        ax.axhline(y=0.1, color='gray', linestyle='--', label='Initial (0.1)')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('lambda_res')
        ax.set_title(f'{ds.capitalize()}')
        ax.legend()
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'Chưa có log', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{ds.capitalize()} (No data)')

plt.tight_layout()
plt.savefig('/kaggle/working/lambda_res_evolution_v2a.png', dpi=150)
plt.show()
print('[ĐÃ LƯU BIỂU ĐỒ] /kaggle/working/lambda_res_evolution_v2a.png')


## Cell 9 — Xuất Kết quả CSV cho Khóa luận


In [ ]:
# Cell 9: Xuất bảng kết quả CSV
import csv, os

OUT_CSV = '/kaggle/working/ablation_enhanced_v2a.csv'
METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

rows = []
for ds in ['baby', 'sports', 'electronics']:
    bl  = BASELINE[ds]
    v1  = V1_RESULTS[ds]
    v2  = v2_results[ds]['metrics'] or {}
    ep  = v2_results[ds]['epoch']
    for m in METRICS:
        bl_v = bl.get(m)
        v1_v = v1.get(m)
        v2_v = v2.get(m)
        delta_v2 = (v2_v - bl_v)/bl_v*100 if (bl_v and v2_v) else None
        rows.append({'dataset': ds.capitalize(), 'metric': m, 'model': 'STAIR-Baseline', 'value': f'{bl_v:.4f}' if bl_v else '', 'best_epoch': ''})
        rows.append({'dataset': ds.capitalize(), 'metric': m, 'model': 'Enhanced-v1 (Replace)', 'value': f'{v1_v:.4f}' if v1_v else '', 'best_epoch': ''})
        rows.append({'dataset': ds.capitalize(), 'metric': m, 'model': 'Enhanced-v2a (Residual)', 'value': f'{v2_v:.4f}' if v2_v else 'N/A', 'best_epoch': str(ep) if ep else 'N/A'})
        if delta_v2 is not None:
            rows.append({'dataset': ds.capitalize(), 'metric': m, 'model': 'Delta v2a (%)', 'value': f'{delta_v2:+.2f}%', 'best_epoch': ''})

with open(OUT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['dataset', 'metric', 'model', 'value', 'best_epoch'])
    writer.writeheader()
    writer.writerows(rows)

print(f'[ĐÃ XUẤT CSV] {OUT_CSV}')
